In [0]:
%run ../utils

In [0]:
dbutils.widgets.text('date_val','')
date_val = dbutils.widgets.get('date_val')

In [0]:
date_val = calculate_date(date_val)

In [0]:
input_path = f'/mnt/bronze-layer/{date_val}/*.json'

df = spark.read.json(input_path)


In [0]:
result_df = df.select('metadata.count')
initial_record_cnt  = result_df.collect()[0]['count']
display(initial_record_cnt)

In [0]:
# Explode the 'features' column to create a new row for each element in the array
df = df.withColumn('features', explode('features'))

# Select the 'properties' field from the exploded 'features' column
df = df.select('features.properties')

In [0]:
print(df.printSchema())

In [0]:
col_lst = ['mag','place','time','updated','tz','url','detail','felt','cdi','mmi','alert','status','tsunami','sig','net','code','ids','sources','types','nst','dmin','rms','gap','magType','type','title']
for column in col_lst:
    df = df.withColumn(column, col(f'properties.{column}'))
df = df.drop('properties')

In [0]:
final_cnt = df.count()

if initial_record_cnt != final_cnt:
    raise ValueError(f"Initial count: {initial_record_cnt} does not match with the final count : {final_cnt}")

In [0]:

df = df.withColumn('time_timestamp', from_unixtime(col('time')/1000)).withColumn('updated_timestamp', from_unixtime(col('updated')/1000)).withColumn("ingest_ts",current_timestamp())

In [0]:
df.write.mode("overwrite").format('parquet').save(f'/mnt/silver-layer/{date_val}/earthquakedata')

In [0]:
display(dbutils.fs.ls("/mnt/silver-layer/2025-05-10/earthquakedata/"))